# Limpeza, Preparação e Armazenamento de Dados — NFL Play by Play

## Decisões de tratamento

- Campos de contexto de uma jogada podem estar ausentes por não se aplicarem ao evento ou por não terem sido registrados. Por isso, não serão preenchidos por média/moda nem serão removidas linhas apenas por conterem valores ausentes.
- Strings vazias ou compostas somente por espaços serão padronizadas como ausentes; espaços no início/fim dos textos serão removidos.
- `Date` será interpretada como data; colunas numéricas e indicadores serão convertidos somente a partir de listas explícitas de atributos conhecidos, sem inferir que identificadores/códigos textuais sejam números.
- Indicadores só serão convertidos para booleano anulável quando seus valores observados forem exclusivamente 0 e 1. Valores fora desse domínio serão reportados e preservados.
- Somente linhas integralmente duplicadas em todas as colunas serão removidas. Possíveis anomalias de domínio serão quantificadas, mas não corrigidas nem excluídas sem regra de negócio que justifique a alteração.

In [ ]:
# Importa as bibliotecas usadas
from pathlib import Path
import numpy as np
import pandas as pd
import kagglehub
from pandas.testing import assert_frame_equal

c:\Users\41228408807\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carregar a fonte de dado

O dataset é obtido pelo mesmo identificador KaggleHub utilizado no notebook `Pre_processamento_dados_Gabarito.ipynb`. O caminho é localizado pelo nome do CSV para não depender de um caminho local específico.

In [ ]:
# Baixa/localiza a versão do dataset e encontra o arquivo esperado
dataset_root = Path(kagglehub.dataset_download("maxhorowitz/nflplaybyplay2009to2016"))
csv_name = "NFL Play by Play 2009-2017 (v4).csv"
csv_matches = list(dataset_root.rglob(csv_name))
if not csv_matches:
    raise FileNotFoundError(f"Arquivo {csv_name!r} não encontrado em {dataset_root}")
csv_path = csv_matches[0]

# Lê sem conversão parcial de tipos para reduzir avisos de tipos mistos.
df_original = pd.read_csv(csv_path, low_memory=False)
df = df_original.copy()
print(f"CSV: {csv_path}")
print(f"Dimensões originais: {df.shape[0]:,} linhas × {df.shape[1]:,} colunas")

CSV: C:\Users\41228408807\.cache\kagglehub\datasets\maxhorowitz\nflplaybyplay2009to2016\versions\6\NFL Play by Play 2009-2017 (v4).csv
Dimensões originais: 407,688 linhas × 102 colunas


## 2. Diagnóstico

As saídas abaixo mostram a estrutura, tipos inferidos na leitura, estatísticas e completude por atributo. O levantamento de strings vazias complementa `isna()`, pois uma célula vazia em um CSV nem sempre é representada como ausente pelo leitor.

In [ ]:
# Inspeciona o df, estrutura e tipos
display(df.head())
df.info(verbose=True)
display(df.describe(include="all").T)
display(df.dtypes.value_counts().rename("quantidade de colunas").to_frame())

,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2009-09-10,2009091000,1,1,<NA>,0 days 00:15:00,15,3600,0,TEN,...,NaN,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,NaN,NaN,2009
1,2009-09-10,2009091000,1,1,1,0 days 00:14:53,15,3593,7,PIT,...,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,2009-09-10,2009091000,1,1,2,0 days 00:14:16,15,3556,37,PIT,...,NaN,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,NaN,NaN,2009
3,2009-09-10,2009091000,1,1,3,0 days 00:13:35,14,3515,41,PIT,...,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2009-09-10,2009091000,1,1,4,0 days 00:13:27,14,3507,8,PIT,...,NaN,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,NaN,NaN,2009


<class 'pandas.DataFrame'>
RangeIndex: 407688 entries, 0 to 407687
Data columns (total 102 columns):
 #    Column                       Dtype          
---   ------                       -----          
 0    Date                         datetime64[us] 
 1    GameID                       Int64          
 2    Drive                        Int64          
 3    qtr                          Int64          
 4    down                         Int64          
 5    time                         timedelta64[us]
 6    TimeUnder                    Int64          
 7    TimeSecs                     Int64          
 8    PlayTimeDiff                 Int64          
 9    SideofField                  string         
 10   yrdln                        Int64          
 11   yrdline100                   Int64          
 12   ydstogo                      Int64          
 13   ydsnet                       Int64          
 14   GoalToGo                     boolean        
 15   FirstDown                 

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Date,407688,NaN,NaN,NaN,2013-11-12 12:32:42.865721,2009-09-10 00:00:00,2011-10-09 00:00:00,2013-11-10 00:00:00,2015-12-13 00:00:00,2017-12-31 00:00:00,NaN
GameID,407688.0,<NA>,<NA>,<NA>,2013157548.708797,2009091000.0,2011100907.0,2013111000.0,2015121301.0,2017123115.0,2572838.560767
Drive,407688.0,<NA>,<NA>,<NA>,12.316158,1.0,6.0,12.0,18.0,35.0,7.149527
qtr,407688.0,<NA>,<NA>,<NA>,2.577412,1.0,2.0,3.0,4.0,5.0,1.12975
down,346534.0,<NA>,<NA>,<NA>,2.002476,1.0,1.0,2.0,3.0,4.0,1.006353
...,...,...,...,...,...,...,...,...,...,...,...
Win_Prob,382679.0,NaN,NaN,NaN,0.50132,0.0,0.276472,0.50447,0.725477,1.0,0.287445
WPA,402147.0,NaN,NaN,NaN,0.002099,-0.997214,-0.014728,0.0,0.014684,0.994848,0.045363
airWPA,159187.0,NaN,NaN,NaN,0.015135,-0.999881,-0.011518,0.003441,0.035792,0.994848,0.05649
yacWPA,158926.0,NaN,NaN,NaN,-0.01048,-0.986673,-0.018683,0.0,0.011431,1.0,0.068139


,quantidade de colunas
string,36
int64,19
Float64,15
Int64,14
float64,9
boolean,7
datetime64[us],1
timedelta64[us],1


In [13]:
# Resume valores ausentes, incluindo proporção por coluna.
missing_initial = pd.DataFrame({
    "ausentes": df.isna().sum(),
    "percentual": df.isna().mean().mul(100),
}).sort_values("ausentes", ascending=False)
display(missing_initial)

,ausentes,percentual
DefTwoPoint,407664,99.994113
BlockingPlayer,407571,99.971302
TwoPointConv,407083,99.851602
ChalReplayResult,404286,99.165538
RecFumbPlayer,403315,98.927366
...,...,...
HomeTeam,0,0.000000
Timeout_Indicator,0,0.000000
TwoPoint_Prob,0,0.000000
ExPoint_Prob,0,0.000000


In [ ]:
# Conta textos vazios ou compostos somente por espaços nas colunas textuais.
text_columns_initial = df.select_dtypes(include=["object", "string"]).columns
blank_initial = df[text_columns_initial].apply(
    lambda column: column.astype("string").str.strip().eq("").sum()
)

display(blank_initial[blank_initial.gt(0)].rename("strings vazias/brancas").to_frame())

display(df.nunique(dropna=True).sort_values(ascending=False).rename("valores distintos").to_frame())
exact_duplicates_initial = int(df.duplicated().sum())

print(f"Linhas integralmente duplicadas: {exact_duplicates_initial:,}")

,strings vazias/brancas


,valores distintos
desc,375962
WPA,366549
Away_WP_post,363360
Home_WP_post,363190
Away_WP_pre,362562
...,...
ChalReplayResult,2
Timeout_Indicator,2
InterceptionThrown,2
TwoPoint_Prob,2


Linhas integralmente duplicadas: 0


## Nomes das colunas

A célula seguinte imprime cada nome de coluna, numerado e sem truncamento, para facilitar a elaboração de um dicionário de dados.

In [ ]:
# Lista todos os nomes das colunas para consulta no dicionário de dados.
for position, column_name in enumerate(df.columns, start=1):
    print(f"{position:03d}. {column_name}")

## 3. Tipificação e padronização textual

As listas seguintes representam atributos cuja semântica indica número ou indicador. A conversão é limitada às colunas presentes no arquivo. Números são armazenados em tipos anuláveis (`Int64` ou `Float64`), que preservam ausências. Quando um atributo numérico conhecido contém algum valor não conversível, o valor passa a ausente e a quantidade de conversões malsucedidas é registrada; revise esse relatório antes de interpretar o resultado. O campo `time` representa o relógio do período em cada jogada; será convertido de `MM:SS` para segundos decorridos no relógio registrado e armazenado como `Int16` anulável.

In [4]:
# Remove espaços externos e transforma textos vazios em ausentes.
text_columns = df.select_dtypes(include=["object", "string"]).columns
for column in text_columns:
    cleaned = df[column].astype("string").str.strip()
    df[column] = cleaned.mask(cleaned.eq(""), pd.NA)

# Converte a data do jogo e contabiliza textos não ausentes que falharam.
invalid_date_count = 0
if "Date" in df.columns:
    date_before = df["Date"]
    parsed_date = pd.to_datetime(date_before, errors="coerce")
    invalid_date_count = int((date_before.notna() & parsed_date.isna()).sum())
    df["Date"] = parsed_date
print(f"Valores não ausentes de Date que não puderam ser interpretados: {invalid_date_count:,}")

# Tipifica atributos numéricos conhecidos, sem converter identificadores textuais.
numeric_candidates = [
    "GameID", "Drive", "qtr", "down", "TimeUnder", "TimeSecs",
    "PlayTimeDiff", "yrdln", "yrdline100", "ydstogo", "ydsnet",
    "Yards.Gained", "posteam_score", "Def_score",
    "ScoreDiff", "AbsScoreDiff", "posteam_timeouts_remaining",
    "DefensiveTeam_timeouts_remaining",
    "No_Score_Prob", "Opp_Field_Goal_Prob", "Opp_Safety_Prob",
    "Opp_Touchdown_Prob", "Field_Goal_Prob", "Safety_Prob",
    "Touchdown_Prob", "ExPoint_Prob", "TwoPoint_Prob", "ExpPts",
    "EPA", "air_yards", "yards_after_catch", "Home_WP_pre",
    "Home_WP_post", "Away_WP_pre", "Away_WP_post",
]
numeric_conversion_report = {}
for column in numeric_candidates:
    if column not in df.columns:
        continue
    original_non_missing = df[column].notna()
    converted = pd.to_numeric(df[column], errors="coerce")
    numeric_conversion_report[column] = int((original_non_missing & converted.isna()).sum())
    values = converted.dropna()
    if values.empty or np.equal(np.mod(values, 1), 0).all():
        df[column] = converted.astype("Int64")
    else:
        df[column] = converted.astype("Float64")
display(pd.Series(numeric_conversion_report, name="valores numéricos inválidos"))

# Converte indicadores somente quando os valores observados estão em {0, 1}.
indicator_candidates = [
    "GoalToGo", "FirstDown", "PlayAttempted", "sp", "Touchdown",
    "ExPointSuccessful", "TwoPointConv", "DefTwoPoint", "Safety",
    "Onsidekick", "Punt", "field_goal_attempt",
    "kickoff_attempt", "fumble", "fumble_forced", "fumble_notforced",
    "fumble_oob", "solo_tackle", "assist_tackle", "qb_hit", "rush_attempt",
    "pass_attempt", "sack", "interception",
    "complete_pass", "incomplete_pass", "lateral_reception",
    "lateral_rush", "lateral_return", "fumble_lost", "touchback",
    "own_kickoff", "own_kickoff_recovery", "return", "penalty",
    "tackled_for_loss"
]
indicator_domain_issues = {}
for column in indicator_candidates:
    if column not in df.columns:
        continue
    numeric_values = pd.to_numeric(df[column], errors="coerce")
    invalid = df[column].notna() & (numeric_values.isna() | ~numeric_values.isin([0, 1]))
    invalid_count = int(invalid.sum())
    if invalid_count:
        indicator_domain_issues[column] = invalid_count
    else:
        df[column] = numeric_values.astype("boolean")
display(pd.Series(indicator_domain_issues, name="indicadores fora de {0, 1}"))

# Converte o relógio MM:SS em segundos e registra valores inválidos.
invalid_time_count = 0
if "time" in df.columns:
    time_before = df["time"]
    parsed_time = pd.to_timedelta("00:" + time_before.astype("string"), errors="coerce")
    invalid_time_count = int((time_before.notna() & parsed_time.isna()).sum())
    total_seconds = parsed_time.dt.total_seconds()
    fractional_seconds = total_seconds.notna() & total_seconds.mod(1).ne(0)
    if fractional_seconds.any():
        raise ValueError("A coluna 'time' contém valores com frações de segundo não esperadas.")
    valid_seconds = total_seconds.dropna()
    if not valid_seconds.between(0, np.iinfo(np.int16).max).all():
        raise ValueError("A coluna 'time' contém segundos fora do intervalo representável por Int16.")
    df["time"] = total_seconds.astype("Int16")
print(f"Valores não ausentes de time que não puderam ser interpretados: {invalid_time_count:,}")

Valores não ausentes de Date que não puderam ser interpretados: 0


GameID                 0
Drive                  0
qtr                    0
down                   0
TimeUnder              0
TimeSecs               0
PlayTimeDiff           0
yrdln                  0
yrdline100             0
ydstogo                0
ydsnet                 0
Yards.Gained           0
ScoreDiff              0
AbsScoreDiff           0
No_Score_Prob          0
Opp_Field_Goal_Prob    0
Opp_Safety_Prob        0
Opp_Touchdown_Prob     0
Field_Goal_Prob        0
Safety_Prob            0
Touchdown_Prob         0
ExPoint_Prob           0
TwoPoint_Prob          0
ExpPts                 0
EPA                    0
Home_WP_pre            0
Home_WP_post           0
Away_WP_pre            0
Away_WP_post           0
Name: valores numéricos inválidos, dtype: int64

TwoPointConv    605
DefTwoPoint      24
Name: indicadores fora de {0, 1}, dtype: int64

Valores não ausentes de time que não puderam ser interpretados: 0


## 4. Valores ausentes

Não se imputam valores nos atributos das jogadas: sem contexto do significado de cada ausência, substituir por zero, média ou moda pode criar eventos fictícios (por exemplo, confundir “não se aplica” com uma medição igual a zero). Também não se excluem registros apenas por conterem ausências. Textos vazios/brancos são normalizados como ausentes. Datas e tempos inválidos são convertidos para ausentes e quantificados. Para a coluna numérica `time`, os ausentes permanecem como nulos no tipo anulável `Int16`; não é possível inserir `""` mantendo essa coluna como inteiro. No Parquet, nulos são representados como valores null com metadados de validade, preservando o tipo numérico e evitando converter a coluna em texto, o que poderia aumentar o arquivo. A tabela abaixo permite comparar as ausências antes e depois do tratamento.

In [5]:
# Compara contagem e percentual de ausências antes e depois da padronização.
missing_after_typing = pd.DataFrame({
    "ausentes antes": df_original.isna().sum(),
    "ausentes depois": df.isna().sum(),
    "percentual depois": df.isna().mean().mul(100),
}).sort_values("ausentes depois", ascending=False)
display(missing_after_typing)
print(f"Valores inválidos de data convertidos para ausentes: {invalid_date_count:,}")
print(f"Valores inválidos de time convertidos para ausentes: {invalid_time_count:,}")

,ausentes antes,ausentes depois,percentual depois
DefTwoPoint,407664,407664,99.994113
BlockingPlayer,407571,407571,99.971302
TwoPointConv,407083,407083,99.851602
ChalReplayResult,404286,404286,99.165538
RecFumbPlayer,403315,403315,98.927366
...,...,...,...
HomeTeam,0,0,0.000000
Timeout_Indicator,0,0,0.000000
TwoPoint_Prob,0,0,0.000000
ExPoint_Prob,0,0,0.000000


Valores inválidos de data convertidos para NaT: 0


## 5. Duplicidades e verificações de consistência

Duplicatas serão definidas como linhas idênticas em todas as colunas e removidas mantendo a primeira ocorrência. Verificações de domínio simples servem para sinalizar itens que merecem revisão, não para presumir que os dados sejam incorretos. O trimestre pode incluir overtime (até 5 no domínio esperado desta fonte), o down deve estar entre 1 e 4 e `ydstogo` não deve ser negativo. Nenhum registro será excluído por esses alertas.

In [6]:
# Registra duplicatas integrais e remove apenas repetições exatas.
duplicate_count_removed = int(df.duplicated().sum())
df = df.drop_duplicates(keep="first").reset_index(drop=True)

# Quantifica valores fora dos domínios simples; ausências não são contadas.
domain_issue_counts = {}
domain_rules = {
    "qtr fora de 1..5": ("qtr", lambda values: ~values.between(1, 5)),
    "down fora de 1..4": ("down", lambda values: ~values.between(1, 4)),
    "ydstogo negativo": ("ydstogo", lambda values: values.lt(0)),
}
for label, (column, invalid_rule) in domain_rules.items():
    if column in df.columns:
        numeric_values = pd.to_numeric(df[column], errors="coerce")
        domain_issue_counts[label] = int((numeric_values.notna() & invalid_rule(numeric_values)).sum())
display(pd.Series(domain_issue_counts, name="quantidade sinalizada"))
print(f"Duplicatas integrais removidas: {duplicate_count_removed:,}")
print(f"Dimensões após limpeza: {df.shape[0]:,} linhas × {df.shape[1]:,} colunas")

qtr fora de 1..5     0
down fora de 1..4    0
ydstogo negativo     0
Name: quantidade sinalizada, dtype: int64

Duplicatas integrais removidas: 0
Dimensões após limpeza: 407,688 linhas × 102 colunas


## 6. Resumo da limpeza e dos tipos finais

A tabela resume alterações estruturais e a completude final. As inconsistências sinalizadas permanecem no DataFrame para evitar correções sem evidência ou regra de negócio.

In [7]:
# Consolida indicadores finais antes da persistência.
quality_summary = pd.Series({
    "linhas originais": len(df_original),
    "linhas finais": len(df),
    "colunas": df.shape[1],
    "duplicatas exatas removidas": duplicate_count_removed,
    "valores ausentes finais": int(df.isna().sum().sum()),
    "valores de Date inválidos": invalid_date_count,
    "indicadores com valores fora de {0, 1}": len(indicator_domain_issues),
    "regras de domínio com ocorrências": sum(count > 0 for count in domain_issue_counts.values()),
}, name="resultado")
display(quality_summary.to_frame())
display(pd.DataFrame({"tipo final": df.dtypes.astype(str), "ausentes": df.isna().sum()}))

,resultado
linhas originais,407688
linhas finais,407688
colunas,102
duplicatas exatas removidas,0
valores ausentes finais,11505187
valores de Date inválidos,0
"indicadores com valores fora de {0, 1}",2
regras de domínio com ocorrências,0


,tipo final,ausentes
Date,datetime64[us],0
GameID,Int64,0
Drive,Int64,0
qtr,Int64,0
down,Int64,61154
...,...,...
Win_Prob,float64,25009
WPA,float64,5541
airWPA,float64,248501
yacWPA,float64,248762


## 7. Armazenamento em Parquet e comparação de tamanho

O arquivo tratado será salvo no diretório de trabalho atual. Parquet é colunar e costuma ser menor que CSV, mas o resultado depende dos dados, compressão e tipos. A comparação será feita em bytes e também apresentada em MB e percentual; não se presume previamente qual arquivo será menor.

In [8]:
# Grava sem o índice do pandas, pois ele não representa um atributo do dataset.
parquet_path = Path.cwd() / "NFL_play_by_play_tratado.parquet"
df.to_parquet(parquet_path, index=False)

# Compara o tamanho físico do CSV de entrada com o Parquet gerado.
csv_size_bytes = csv_path.stat().st_size
parquet_size_bytes = parquet_path.stat().st_size
size_reduction_percent = (1 - parquet_size_bytes / csv_size_bytes) * 100 if csv_size_bytes else float("nan")
size_comparison = pd.DataFrame({
    "arquivo": [str(csv_path), str(parquet_path)],
    "tamanho (bytes)": [csv_size_bytes, parquet_size_bytes],
    "tamanho (MB)": [csv_size_bytes / (1024 ** 2), parquet_size_bytes / (1024 ** 2)],
})
display(size_comparison)
print(f"Redução do tamanho: {size_reduction_percent:.2f}% (valor negativo indica aumento).")

,arquivo,tamanho (bytes),tamanho (MB)
0,C:\Users\41228408807\.cache\kagglehub\datasets...,275530460,262.766323
1,c:\Users\41228408807\Documents\GitHub\uniSENAI...,76534261,72.988759


Redução do tamanho: 72.22% (valor negativo indica aumento).


## 8. Validação após armazenamento

O Parquet é reaberto e comparado com o DataFrame tratado, incluindo ordem das colunas e conteúdo, com ausências equivalentes. O índice não é comparado porque foi deliberadamente excluído do arquivo. A mensagem de validação só aparece se as verificações forem aprovadas; qualquer divergência gera uma exceção explícita.

In [9]:
# Recarrega o arquivo persistido e valida dimensões, colunas, tipos e valores.
df_reloaded = pd.read_parquet(parquet_path)
assert df_reloaded.shape == df.shape, "As dimensões mudaram após a gravação Parquet."
assert df_reloaded.columns.tolist() == df.columns.tolist(), "A ordem ou os nomes das colunas mudaram."
assert_frame_equal(
    df.reset_index(drop=True),
    df_reloaded.reset_index(drop=True),
    check_dtype=True,
    check_categorical=False,
    check_exact=False,
    check_names=True,
)
print("Validação aprovada: dimensões, colunas e dados foram preservados no Parquet.")

Validação aprovada: dimensões, colunas e dados foram preservados no Parquet.


## 9. Análise final

O resumo final é produzido a partir dos resultados calculados durante a execução, sem números pré-preenchidos. Ele reúne as principais mudanças, caminhos e comparação de armazenamento.

In [10]:
# Apresenta os principais resultados para documentar a execução.
final_summary = {
    "CSV de origem": str(csv_path),
    "Parquet tratado": str(parquet_path.resolve()),
    "linhas antes": len(df_original),
    "linhas após limpeza": len(df),
    "colunas preservadas": df.shape[1],
    "duplicatas integrais removidas": duplicate_count_removed,
    "ausências mantidas como nulos no resultado": int(df.isna().sum().sum()),
    "datas inválidas convertidas para NaT": invalid_date_count,
    "tempos de jogo inválidos convertidos para ausentes": invalid_time_count,
    "time armazenado como segundos em Int16 anulável": "sim" if "time" in df.columns else "coluna ausente",
    "conversões numéricas malsucedidas": sum(numeric_conversion_report.values()),
    "valores fora do domínio de indicadores": sum(indicator_domain_issues.values()),
    "ocorrências sinalizadas pelas regras de domínio": sum(domain_issue_counts.values()),
    "tamanho CSV (MB)": round(csv_size_bytes / (1024 ** 2), 2),
    "tamanho Parquet (MB)": round(parquet_size_bytes / (1024 ** 2), 2),
    "redução de tamanho (%)": round(size_reduction_percent, 2),
    "validação Parquet": "aprovada se esta célula foi concluída sem exceção",
}
display(pd.Series(final_summary, name="resultado final").to_frame())
print(
    "Tratamento concluído: tipos semânticos ajustados, vazios padronizados, "
    "duplicatas exatas removidas, anomalias reportadas sem exclusão e Parquet validado."
)

,resultado final
CSV de origem,C:\Users\41228408807\.cache\kagglehub\datasets...
Parquet tratado,C:\Users\41228408807\Documents\GitHub\uniSENAI...
linhas antes,407688
linhas após limpeza,407688
colunas preservadas,102
duplicatas integrais removidas,0
ausências mantidas no resultado,11505187
datas inválidas convertidas para NaT,0
tempos de jogo inválidos convertidos para NaT,0
conversões numéricas malsucedidas,0


Tratamento concluído: tipos semânticos ajustados, vazios padronizados, duplicatas exatas removidas, anomalias reportadas sem exclusão e Parquet validado.
